# Tabular Machine Learning Models Training
**Project**: Horizon- and Regime-Specific ML for Stock Return Prediction  
**Market**: Indian Equity (NSE) exclusively  
**Run ID**: tabular_ml_models  

## Models
- Naive Baseline (majority-class)
- Logistic Regression
- Random Forest
- XGBoost

## Structure
1. INPUTS / CONFIGURATION
2. Environment Setup
3. Data Loading & Validation
4. Walk-Forward Training Loop
5. Per-Regime Metric Breakdown
6. Results Aggregation
7. Checkpoint Summary
8. Final ZIP Export

## 1. INPUTS / CONFIGURATION
Edit these paths to match your Kaggle dataset mount point.

In [ ]:
# --- CELL 1: CONFIGURATION + AUTO-DETECT PATHS -------------------------
# Automatically finds configs/ and data/processed/ regardless of how
# Kaggle nests the dataset. No manual path editing needed.

import os

def find_project_dirs(input_root='/kaggle/input'):
    """Walk /kaggle/input and find the configs/ and data/processed/ dirs."""
    config_dir = None
    data_dir   = None

    for root, dirs, files in os.walk(input_root):
        # Stop descending too deep (safety)
        depth = root.replace(input_root, '').count(os.sep)
        if depth > 5:
            dirs[:] = []
            continue

        # Look for configs/ folder containing experiment.yaml
        if config_dir is None and 'configs' in dirs:
            candidate = os.path.join(root, 'configs')
            if os.path.isfile(os.path.join(candidate, 'experiment.yaml')):
                config_dir = candidate

        # Look for data/processed/ folder containing features_targets.csv
        if data_dir is None and 'processed' in dirs:
            candidate = os.path.join(root, 'processed')
            if os.path.isfile(os.path.join(candidate, 'features_targets.csv')):
                data_dir = candidate

        if config_dir and data_dir:
            break

    return config_dir, data_dir


CONFIG_DIR, DATA_DIR = find_project_dirs()

if CONFIG_DIR is None or DATA_DIR is None:
    # Diagnostic: print actual directory tree so the user can fix it
    print('ERROR: Could not auto-detect required directories.')
    print('Actual contents of /kaggle/input (max depth 4):')
    for root, dirs, files in os.walk('/kaggle/input'):
        depth = root.replace('/kaggle/input', '').count(os.sep)
        if depth > 4:
            dirs[:] = []
            continue
        indent = '  ' * depth
        print(f'{indent}{os.path.basename(root)}/')
        for fname in files[:5]:   # show first 5 files per folder
            print(f'{indent}  {fname}')
    raise RuntimeError(
        'Dataset not found. Make sure you added nse-ml-project via '
        'Add Data on the right panel, then restart and run all cells.'
    )

OUTPUT_DIR  = '/kaggle/working/tabular_ml_output'
RUN_ID      = 'tabular_ml_models'
RANDOM_SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Paths detected successfully')
print(f'  CONFIG_DIR : {CONFIG_DIR}')
print(f'  DATA_DIR   : {DATA_DIR}')
print(f'  OUTPUT_DIR : {OUTPUT_DIR}')

# Quick sanity check: verify key files exist
required = [
    os.path.join(CONFIG_DIR, 'experiment.yaml'),
    os.path.join(CONFIG_DIR, 'models.yaml'),
    os.path.join(DATA_DIR, 'features_targets.csv'),
    os.path.join(DATA_DIR, 'feature_list.txt'),
]
all_ok = True
for path in required:
    exists = os.path.isfile(path)
    status = 'OK' if exists else 'MISSING'
    print(f'  [{status}] {os.path.basename(path)}')
    if not exists:
        all_ok = False

if not all_ok:
    raise FileNotFoundError('One or more required files are missing — check dataset upload.')
print('All required files present. Ready to run.')

## 2. Environment Setup

In [ ]:
!pip install xgboost scikit-learn pandas numpy pyyaml --quiet

import warnings
warnings.filterwarnings('ignore')

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, log_loss
)
from xgboost import XGBClassifier

print('Libraries loaded.')

## 3. Data Loading & Validation

In [ ]:
# ── Load experiment config ──────────────────────────────────────────────────
with open(f'{CONFIG_DIR}/experiment.yaml') as f:
    exp_cfg = yaml.safe_load(f)

with open(f'{CONFIG_DIR}/models.yaml') as f:
    mdl_cfg = yaml.safe_load(f)

SPLITS_CFG = exp_cfg['walk_forward']['splits']
HORIZONS   = exp_cfg['horizons']              # [1, 5]
BENCHMARK  = exp_cfg['data']['benchmark']     # '^NSEI'

print(f'Splits: {len(SPLITS_CFG)}  |  Horizons: {HORIZONS}')

In [ ]:
# ── Load feature+target dataset ─────────────────────────────────────────────
df = pd.read_csv(f'{DATA_DIR}/features_targets.csv', parse_dates=['Date'])

# Feature list
with open(f'{DATA_DIR}/feature_list.txt') as f:
    FEATURE_COLS = [l.strip() for l in f if l.strip()]

# Exclude benchmark from modelling
df_stocks = df[df['Ticker'] != BENCHMARK].copy().reset_index(drop=True)

print(f'Stocks df: {df_stocks.shape}  |  Features: {len(FEATURE_COLS)}')
print(f'Tickers: {sorted(df_stocks["Ticker"].unique())}')
print(f'Date range: {df_stocks["Date"].min().date()} -> {df_stocks["Date"].max().date()}')

In [ ]:
# ── Unique sorted trading dates (for purge calculation) ─────────────────────
# Cast to numpy datetime64[ns] so np.searchsorted works with pd.Timestamp args
ALL_DATES = np.sort(df_stocks['Date'].values.astype('datetime64[ns]'))
EMBARGO_DAYS = 5
REGIME_NAMES = {0: 'Low', 1: 'Medium', 2: 'High'}

def get_purge_cutoff(test_start, horizon):
    # Convert pd.Timestamp -> datetime64[ns] for numpy compatibility
    ts_np  = np.datetime64(test_start, 'ns')
    ts_idx = np.searchsorted(ALL_DATES, ts_np, side='left')
    purge_idx = max(ts_idx - horizon, 0)
    return pd.Timestamp(ALL_DATES[purge_idx])

def get_embargo_cutoff(purge_cutoff, embargo_days):
    pc_np  = np.datetime64(purge_cutoff, 'ns')
    pc_idx = np.searchsorted(ALL_DATES, pc_np, side='right')
    emb_idx = min(pc_idx + embargo_days - 1, len(ALL_DATES) - 1)
    return pd.Timestamp(ALL_DATES[emb_idx])

def clean(X):
    """Replace inf/NaN with 0."""
    return np.where(np.isfinite(X), X, 0.0)

print('Helper functions defined.')

## 4. Walk-Forward Training Loop

In [ ]:
# ── Model factory ───────────────────────────────────────────────────────────
def make_models(mdl_cfg, seed=42):
    lr_cfg = mdl_cfg['logistic_regression']
    rf_cfg = mdl_cfg['random_forest']
    xgb_cfg = mdl_cfg['xgboost']

    return {
        'Naive': DummyClassifier(strategy='most_frequent', random_state=seed),
        'LogReg': LogisticRegression(
            C=lr_cfg['C'], max_iter=lr_cfg['max_iter'],
            solver=lr_cfg['solver'], class_weight=lr_cfg['class_weight'],
            random_state=seed
        ),
        'RF': RandomForestClassifier(
            n_estimators=rf_cfg['n_estimators'], max_depth=rf_cfg['max_depth'],
            min_samples_leaf=rf_cfg['min_samples_leaf'],
            max_features=rf_cfg['max_features'],
            class_weight=rf_cfg['class_weight'],
            n_jobs=-1, random_state=seed
        ),
        'XGB': XGBClassifier(
            n_estimators=xgb_cfg['n_estimators'], max_depth=xgb_cfg['max_depth'],
            learning_rate=xgb_cfg['learning_rate'],
            subsample=xgb_cfg['subsample'],
            colsample_bytree=xgb_cfg['colsample_bytree'],
            min_child_weight=xgb_cfg['min_child_weight'],
            reg_alpha=xgb_cfg['reg_alpha'], reg_lambda=xgb_cfg['reg_lambda'],
            eval_metric='logloss', verbosity=0,
            random_state=seed, n_jobs=-1
        ),
    }

print('Model factory defined.')

In [ ]:
# ── Metric calculator ───────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_prob, label_prefix=''):
    """Compute classification metrics; handle degenerate cases."""
    n = len(y_true)
    if n == 0:
        return {}
    
    metrics = {
        f'{label_prefix}n': n,
        f'{label_prefix}accuracy': round(accuracy_score(y_true, y_pred), 4),
        f'{label_prefix}f1': round(f1_score(y_true, y_pred, zero_division=0), 4),
        f'{label_prefix}precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        f'{label_prefix}recall': round(recall_score(y_true, y_pred, zero_division=0), 4),
    }
    
    # AUC-ROC only if both classes present
    if len(np.unique(y_true)) == 2 and y_prob is not None:
        metrics[f'{label_prefix}roc_auc'] = round(roc_auc_score(y_true, y_prob), 4)
        metrics[f'{label_prefix}log_loss'] = round(log_loss(y_true, y_prob), 4)
    else:
        metrics[f'{label_prefix}roc_auc'] = None
        metrics[f'{label_prefix}log_loss'] = None
    
    return metrics

print('Metric calculator defined.')

In [ ]:
# ── Main walk-forward training loop ────────────────────────────────────────
all_results   = []   # One row per model × split × horizon
all_preds     = []   # Row-level predictions for backtest (Phase 7 portfolio backtest)

total_combos = len(SPLITS_CFG) * len(HORIZONS)
combo_idx = 0

for split_cfg in SPLITS_CFG:
    label      = split_cfg['label']
    train_start = pd.Timestamp(split_cfg['train_start'])
    train_end   = pd.Timestamp(split_cfg['train_end'])
    test_start  = pd.Timestamp(split_cfg['test_start'])
    test_end    = pd.Timestamp(split_cfg['test_end'])
    regime_col  = f'regime_{label}'

    for horizon in HORIZONS:
        combo_idx += 1
        t0 = time.time()
        target_col = f'fwd_label_{horizon}d'
        return_col = f'fwd_return_{horizon}d'

        print(f'\n[{combo_idx}/{total_combos}] {label}  h={horizon}d', flush=True)

        # ── Purge + Embargo ─────────────────────────────────────────────────
        purge_cutoff   = get_purge_cutoff(test_start, horizon)
        embargo_cutoff = get_embargo_cutoff(purge_cutoff, EMBARGO_DAYS)

        train_mask = (
            (df_stocks['Date'] >= train_start) &
            (df_stocks['Date'] <= purge_cutoff) &
            df_stocks[target_col].notna()
        )
        test_mask = (
            (df_stocks['Date'] >= test_start) &
            (df_stocks['Date'] <= test_end) &
            df_stocks[target_col].notna()
        )

        df_train = df_stocks[train_mask].copy()
        df_test  = df_stocks[test_mask].copy()

        print(f'  Train: {len(df_train):,} rows | Test: {len(df_test):,} rows', flush=True)

        if len(df_train) < 100 or len(df_test) < 10:
            print('  SKIP: insufficient data', flush=True)
            continue

        # ── Prepare arrays ──────────────────────────────────────────────────
        X_train_raw = df_train[FEATURE_COLS].values
        X_test_raw  = df_test[FEATURE_COLS].values
        y_train     = df_train[target_col].values.astype(int)
        y_test      = df_test[target_col].values.astype(int)

        # Clean inf/NaN
        X_train_raw = clean(X_train_raw)
        X_test_raw  = clean(X_test_raw)

        # Scale — fit on training ONLY
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train_raw)
        X_test  = scaler.transform(X_test_raw)

        # Regime labels for test
        test_regime = df_test[regime_col].values if regime_col in df_test.columns else np.full(len(df_test), -1)

        # ── Train + evaluate each model ─────────────────────────────────────
        models = make_models(mdl_cfg, seed=RANDOM_SEED)

        for model_name, model in models.items():
            # Naive uses unscaled features (doesn't matter — no features used)
            X_tr = X_train_raw if model_name == 'Naive' else X_train
            X_te = X_test_raw  if model_name == 'Naive' else X_test

            model.fit(X_tr, y_train)

            y_pred = model.predict(X_te)
            try:
                y_prob = model.predict_proba(X_te)[:, 1]
            except Exception:
                y_prob = None

            # ── Overall metrics ─────────────────────────────────────────────
            overall = compute_metrics(y_test, y_pred, y_prob)

            row = {
                'run_id': RUN_ID,
                'split': label,
                'horizon': horizon,
                'model': model_name,
                'train_rows': len(df_train),
                'test_rows': len(df_test),
                'train_start': str(train_start.date()),
                'train_end': str(train_end.date()),
                'test_start': str(test_start.date()),
                'test_end': str(test_end.date()),
                **overall,
            }

            # ── Per-regime metrics ──────────────────────────────────────────
            for regime_id, regime_name in REGIME_NAMES.items():
                mask = test_regime == regime_id
                if mask.sum() < 5:
                    continue
                r_prob = y_prob[mask] if y_prob is not None else None
                regime_metrics = compute_metrics(
                    y_test[mask], y_pred[mask], r_prob,
                    label_prefix=f'regime_{regime_name.lower()}_'
                )
                row.update(regime_metrics)

            all_results.append(row)

            # ── Store row-level predictions for backtest ────────────────────
            pred_df = df_test[['Date', 'Ticker', return_col, target_col, regime_col]].copy()
            pred_df['split']   = label
            pred_df['horizon'] = horizon
            pred_df['model']   = model_name
            pred_df['y_pred']  = y_pred
            pred_df['y_prob']  = y_prob if y_prob is not None else np.nan
            all_preds.append(pred_df)

            acc = overall.get('accuracy', 'n/a')
            auc = overall.get('roc_auc', 'n/a')
            print(f'  {model_name:8s}  acc={acc}  auc={auc}', flush=True)

        elapsed = time.time() - t0
        print(f'  Done in {elapsed:.1f}s', flush=True)

print('\n=== Walk-forward loop complete ===')

## 5. Results Aggregation

In [ ]:
# ── Aggregate results into DataFrames ───────────────────────────────────────
results_df = pd.DataFrame(all_results)
preds_df   = pd.concat(all_preds, ignore_index=True)

print('Results shape:', results_df.shape)
print('Predictions shape:', preds_df.shape)
print(results_df[['split','horizon','model','accuracy','roc_auc','f1']].to_string(index=False))

In [ ]:
# ── Summary pivot: accuracy by model × horizon ──────────────────────────────
pivot_acc = results_df.pivot_table(
    index=['model'], columns=['horizon'], values='accuracy',
    aggfunc='mean'
).round(4)

print('\nMean Accuracy (averaged over splits):')
print(pivot_acc)

pivot_auc = results_df.pivot_table(
    index=['model'], columns=['horizon'], values='roc_auc',
    aggfunc='mean'
).round(4)

print('\nMean ROC-AUC (averaged over splits):')
print(pivot_auc)

In [ ]:
# ── Per-regime accuracy summary ──────────────────────────────────────────────
for regime in ['low', 'medium', 'high']:
    col = f'regime_{regime}_accuracy'
    if col in results_df.columns:
        pivot = results_df.pivot_table(
            index='model', columns='horizon', values=col, aggfunc='mean'
        ).round(4)
        print(f'\n{regime.title()} Vol Regime — Mean Accuracy:')
        print(pivot)

## 6. Save Results

In [ ]:
# ── Save all outputs ─────────────────────────────────────────────────────────
results_path = f'{OUTPUT_DIR}/results_all_splits.csv'
preds_path   = f'{OUTPUT_DIR}/predictions_all_splits.csv'
pivot_path   = f'{OUTPUT_DIR}/accuracy_pivot.csv'
auc_path     = f'{OUTPUT_DIR}/auc_pivot.csv'

results_df.to_csv(results_path, index=False)
preds_df.to_csv(preds_path, index=False)
pivot_acc.to_csv(pivot_path)
pivot_auc.to_csv(auc_path)

print(f'Saved: {results_path}')
print(f'Saved: {preds_path}')
print(f'Saved: {pivot_path}')
print(f'Saved: {auc_path}')

## 7. Checkpoint Summary

In [ ]:
# ── Checkpoint: key findings ─────────────────────────────────────────────────
print('=' * 60)
print('CHECKPOINT SUMMARY — Tabular ML Model Training')
print('=' * 60)
print(f'Run ID   : {RUN_ID}')
print(f'Models   : {list(make_models(mdl_cfg).keys())}')
print(f'Splits   : {len(SPLITS_CFG)}')
print(f'Horizons : {HORIZONS}')
print(f'Results  : {len(results_df)} rows')
print(f'Preds    : {len(preds_df)} rows')
print()

best_acc = results_df.loc[results_df['accuracy'].idxmax()]
print(f'Best single-split accuracy : {best_acc["accuracy"]}  ({best_acc["model"]} | {best_acc["split"]} | h={best_acc["horizon"]}d)')

print()
print('Mean accuracy by model:')
print(results_df.groupby('model')['accuracy'].mean().sort_values(ascending=False).round(4))

print()
print('NOTE: Results must be interpreted per-split and per-regime.')
print('      Do not cherry-pick best splits. Use full table for reporting.')
print('=' * 60)

## 8. Final ZIP Export

In [ ]:
# ── Create ZIP of all artifacts ──────────────────────────────────────────────
import shutil

zip_path = f'/kaggle/working/{RUN_ID}_artifacts'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)

final_zip = f'{zip_path}.zip'
size_mb = os.path.getsize(final_zip) / 1e6
print(f'ZIP created: {final_zip}  ({size_mb:.2f} MB)')
print()
print('Next steps:')
print('  1. Download the ZIP from Kaggle output')
print(f'  2. Extract to  artifacts/kaggle/{RUN_ID}/')
print('  3. Run portfolio backtest locally using predictions_all_splits.csv')